# Stage-2 Revision Experiments (v2) — Self-Contained Kaggle GPU Runner (`3a`, redesigned `3b`, optional `3c` & `3d`)

> **Round-4 Provenance & Environment Note (`3d`)**:
> 1. **Causal US-Calendar Stage-1 `SPILL` Features**: This v2 notebook loads **exclusively** `spill_features_main_us1677_causal.csv` (SHA-256 `224728802dea162577c638d57a06775574769143b6ce2af1c965debcb4a3cafd`), exported from the exact `Main_US_1677` `log_r2` Stage-1 TVP-VAR (`causal_sigma0=True`, `Sigma_0 = cov(y[:250])`, zero JKSE row drops, zero `.ffill()`). It refuses to run if the file is missing or its SHA-256 mismatches, and aborts if any legacy `tvp_var_dynamic_features*.csv` file is ever resolved.
> 2. **PyTorch / Lightning Environment Difference vs. `v6`**: Kaggle's current runtime ships newer `torch` / `pytorch_lightning` releases than the archived `v6` run. Small numeric shifts in `NOSPILL` runs relative to archived `v6` numbers may partly reflect PyTorch/Lightning library version differences in addition to discarding the first `250` warm-up rows; exact runtime versions are logged into `config_used.json`.
> 3. **Redesigned `3b` (`LogVar` & `SoftplusScaled`)**: Replaces collapsed `Softplus_IdentityScale` and `AbsR_Squared*` with `LogVar` (`log(r^2 + 1e-10)` + causal validation Duan smearing on the `158` validation windows) and `SoftplusScaled` (`r^2 / train_mean_r2`, train-only exog standardization, `Softplus` output head + `1e-8` variance floor, output bias initialized to `log(exp(1.0) - 1)`).

## How to Run on Kaggle
1. **Create / Open Notebook**: Upload `kaggle_3a3b_v2.ipynb` to Kaggle Notebooks.
2. **Accelerator & Internet**:
   - **Accelerator**: `GPU T4 x1` (16 GB VRAM).
   - **Internet**: `ON` (needed only during Section 0 for `pip install -q neuralforecast==3.3.0 arch`).
3. **Add Input Datasets**:
   - Primary dataset path: `${KAGGLE_INPUT_ROOT}/data_clean` (`dataset_deep_learning.csv`).
   - Upload `revision/outputs/spill_features_main_us1677_causal.csv` into your Kaggle dataset or attach it alongside via **Add Input** so `spill_features_main_us1677_causal.csv` is discoverable under `/kaggle/input/`.
   - *(Optional, only if `RUN_3D = True`)*: Upload `revision/outputs/step3d_indodax_rv60m_daily.csv` and attach via **Add Input**.
4. **Configure Flags in Section 0**:
   - Default configuration: `RUN_3A = True`, `RUN_3B = True`, `RUN_3C = False`, `RUN_3D = False` (~8 minutes total on T4).
5. **Run All & Collect Outputs**:
   - Outputs are saved incrementally (per seed) inside `/kaggle/working/outputs_3a3b_v2/` and bundled into `/kaggle/working/kaggle_outputs_3a3b_v2.zip`:
     - `predictions_long.csv`, `predictions_ensemble.csv`, `metrics_by_model.csv`, `alignment_manifest.csv`, `config_used.json`, `negative_forecast_report.csv`, and `per_seed/*.csv`.


## Section 0: Setup, Dependencies, Seeds, Run Flags & Custom `SoftplusScaledNBEATSx`

Installs `neuralforecast==3.3.0` (and `arch`) if needed, prints and logs library versions and GPU info (`3d`), sets `RUN_3A..RUN_3D`, defines deterministic `set_all_seeds(seed)`, defines `SoftplusScaledNBEATSx` (for `3b` V-B), and initializes `CONFIG_USED`.


In [ ]:
import gc
import hashlib
import json
import math
import os
import random
import subprocess
import sys
import time
import traceback
import warnings
import zipfile
from pathlib import Path

warnings.filterwarnings("ignore")

# ------------------------------------------------------------------
# 1. Execution Flags & Progress Bar Controls
# ------------------------------------------------------------------
RUN_3A = True
RUN_3B = True
RUN_3C = False
RUN_3D = False
SHOW_PROGRESS_BAR = True         # Outer tqdm progress bar per section
SHOW_TRAINER_STEP_BAR = False    # Keep False to avoid Lightning log flooding

EXPECTED_SPILL_SHA256 = "224728802dea162577c638d57a06775574769143b6ce2af1c965debcb4a3cafd"
SPILL_FILENAME = "spill_features_main_us1677_causal.csv"

# ------------------------------------------------------------------
# 2. Install & Pin neuralforecast==3.3.0 (Internet ON only for pip)
# ------------------------------------------------------------------
def ensure_packages():
    need_install = False
    try:
        import neuralforecast
        if getattr(neuralforecast, "__version__", None) != "3.3.0":
            need_install = True
    except ImportError:
        need_install = True
    try:
        import arch  # noqa: F401
    except ImportError:
        need_install = True

    if need_install:
        print("Installing pinned packages: neuralforecast==3.3.0 arch ...")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "neuralforecast==3.3.0", "arch"],
            check=True,
        )

ensure_packages()

import numpy as np
import pandas as pd
import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tqdm.auto import tqdm
from neuralforecast import NeuralForecast
from neuralforecast.models import TFT, NBEATSx
from neuralforecast.losses.pytorch import MAE, MSE
import neuralforecast

assert neuralforecast.__version__ == "3.3.0", (
    f"Expected neuralforecast==3.3.0, found {neuralforecast.__version__}"
)

# ------------------------------------------------------------------
# 3. Deterministic Seed Helper & Environment Audit (3d)
# ------------------------------------------------------------------
def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    pl.seed_everything(seed, workers=True)
    os.environ["PYTHONHASHSEED"] = str(seed)

cuda_ok = torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0) if cuda_ok else "CPU (No CUDA GPU)"
gpu_vram_gb = (
    round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 2)
    if cuda_ok else 0.0
)

print("=== SECTION 0: ENVIRONMENT AUDIT ===")
print(f"Python          : {sys.version.split()[0]}")
print(f"PyTorch         : {torch.__version__}")
print(f"Lightning       : {pl.__version__}")
print(f"NeuralForecast  : {neuralforecast.__version__}")
print(f"Pandas / NumPy  : {pd.__version__} / {np.__version__}")
print(f"GPU Device      : {gpu_name} ({gpu_vram_gb} GB VRAM)")
print(f"Flags           : RUN_3A={RUN_3A}, RUN_3B={RUN_3B}, RUN_3C={RUN_3C}, RUN_3D={RUN_3D}")

# ------------------------------------------------------------------
# 4. Shared v6 Hyperparameters, Features & Working Directories
# ------------------------------------------------------------------
IS_KAGGLE = Path("/kaggle").exists()
OUTPUT_DIR = Path("/kaggle/working/outputs_3a3b_v2") if IS_KAGGLE else Path("revision/kaggle_outputs/outputs_3a3b_v2")
PER_SEED_DIR = OUTPUT_DIR / "per_seed"
ZIP_PATH = Path("/kaggle/working/kaggle_outputs_3a3b_v2.zip") if IS_KAGGLE else Path("revision/kaggle_outputs/kaggle_outputs_3a3b_v2.zip")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PER_SEED_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 7, 13, 21, 99]
SEEDS_3C = [42, 7, 13]
SEEDS_3D_TFT = [42, 7, 13]
SEEDS_3D_NBX = [42, 7, 13, 21, 99]

H = 10
WARMUP_DISCARD = 250
MAX_STEPS = 1000
VAL_CHECK_STEPS = 25
PATIENCE = 5
N_VAL = 167
N_TEST_CUTOFFS = 335
BLOCK_SIZE_3C = 60
VARIANCE_FLOOR_3B = 1e-8

NOSPILL_COLS = [
    "ret_saham_nvda", "ret_saham_msft", "ret_saham_tsla", "ret_saham_googl", "ret_saham_meta",
    "ret_crypto_btc_usd", "ret_crypto_eth_usd", "crypto_btc_usd_weekend_return",
    "gap_days", "is_monday", "is_ihsg_holiday",
]
SPILL_ONLY_9 = [
    "tvp_tsi",
    "net_spill_saham_nvda", "net_spill_saham_msft", "net_spill_saham_tsla",
    "net_spill_saham_googl", "net_spill_saham_meta",
    "net_spill_crypto_btc_usd", "net_spill_crypto_eth_usd",
    "spillover_ai_to_btcidr",
]
SPILL_COLS = NOSPILL_COLS + SPILL_ONLY_9
assert "net_spill_crypto_btc_idr_indodax" not in SPILL_COLS, "Target spillover must not be in features!"
FEATURE_SETS = {"NOSPILL": NOSPILL_COLS, "SPILL": SPILL_COLS}

# Exact hyperparameters from results/v6/v6_best_hyperparameters.json (also auto-checked in Section 1)
BEST_HP = {
    "TFT": {"input_size": 50, "learning_rate": 0.001, "hidden_size": 64, "loss_name": "MSE"},
    "NBEATSx": {
        "input_size": 20,
        "learning_rate": 0.0003,
        "mlp_units": [[256, 256], [256, 256], [256, 256]],
        "loss_name": "MAE",
    },
}

LONG_FRAMES = []
NEG_REPORT_ROWS = []
CONFIG_USED = {
    "notebook_version": "v2_round4",
    "environment_note": (
        "PyTorch/Lightning runtime versions differ from archived v6 Kaggle runs; "
        "small shifts in NOSPILL metrics may partly reflect environment differences alongside dropping 250 warmup rows."
    ),
    "flags": {"RUN_3A": RUN_3A, "RUN_3B": RUN_3B, "RUN_3C": RUN_3C, "RUN_3D": RUN_3D},
    "versions": {
        "python": sys.version.split()[0],
        "torch": torch.__version__,
        "pytorch_lightning": pl.__version__,
        "neuralforecast": neuralforecast.__version__,
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "scikit_learn": sklearn.__version__,
    },
    "gpu": {"available": bool(cuda_ok), "device_name": gpu_name, "vram_gb": gpu_vram_gb},
    "data_provenance": {
        "expected_spill_filename": SPILL_FILENAME,
        "expected_spill_sha256": EXPECTED_SPILL_SHA256,
    },
    "protocol": {
        "h": H,
        "warmup_discard": WARMUP_DISCARD,
        "max_steps": MAX_STEPS,
        "val_check_steps": VAL_CHECK_STEPS,
        "early_stop_patience_steps": PATIENCE,
        "n_val": N_VAL,
        "n_test_cutoffs": N_TEST_CUTOFFS,
        "block_size_3c": BLOCK_SIZE_3C,
        "variance_floor_3b": VARIANCE_FLOOR_3B,
    },
    "seeds": {
        "3a": SEEDS,
        "3b": SEEDS,
        "3c": SEEDS_3C,
        "3d_tft": SEEDS_3D_TFT,
        "3d_nbeatsx": SEEDS_3D_NBX,
    },
    "exog_columns": {"NOSPILL": NOSPILL_COLS, "SPILL": SPILL_COLS, "spill_diff_9": SPILL_ONLY_9},
    "hyperparameters": BEST_HP,
    "scalers": {
        "3a": "standard",
        "3b_LogVar": "standard (target log(r^2 + 1e-10) + train-915 standardized exog + causal 158-window val Duan smearing)",
        "3b_SoftplusScaled": "identity on r^2 / train_mean_r2 + train-915 standardized exog + Softplus + 1e-8 variance floor",
        "3c": "standard",
        "3d": "standard",
    },
    "target_precision_checks": [],
    "collapse_warnings": [],
    "timings_sec": {},
    "errors": {},
    "fit_logs": [],
}

set_all_seeds(42)
print(f"Outputs directory ready: {OUTPUT_DIR}")


In [ ]:
import os
# ------------------------------------------------------------------
# 1. Input Path Discovery & Strict SHA-256 Provenance Guard (3a)
# ------------------------------------------------------------------
def find_exact_input_file(filename: str, search_roots):
    assert "tvp_var_dynamic_features" not in filename, (
        f"PROHIBITED LEGACY FILE GUARD TRIGGERED: {filename}"
    )
    for root in search_roots:
        if not root.exists():
            continue
        direct = root / filename
        if direct.exists():
            return direct
    for root in search_roots:
        if not root.exists():
            continue
        matches = sorted(root.rglob(filename))
        if matches:
            return matches[0]
    return None

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fp:
        for chunk in iter(lambda: fp.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()

SEARCH_ROOTS = [
    Path(os.environ.get("KAGGLE_INPUT_ROOT", "./data/clean")),
    Path("/kaggle/input"),
    Path("/kaggle/working"),
    Path("data/clean"),
    Path("revision/outputs"),
    Path("inputs/v5"),
    Path("results/v6"),
]

DL_PATH = find_exact_input_file("dataset_deep_learning.csv", SEARCH_ROOTS)
SPILL_PATH = find_exact_input_file(SPILL_FILENAME, SEARCH_ROOTS)
HAR1_PATH = find_exact_input_file("har_preds_T0_h1_v2.csv", SEARCH_ROOTS)
V6_HP_PATH = find_exact_input_file("v6_best_hyperparameters.json", SEARCH_ROOTS)
RV60M_PATH = find_exact_input_file("step3d_indodax_rv60m_daily.csv", SEARCH_ROOTS)

# Hard guard: ensure NO legacy tvp_var_dynamic_features*.csv file is loaded anywhere
for _p in [DL_PATH, SPILL_PATH, HAR1_PATH, V6_HP_PATH, RV60M_PATH]:
    if _p is not None and "tvp_var_dynamic_features" in str(_p):
        raise RuntimeError(f"ABORT: Prohibited legacy TVP feature path resolved: {_p}")

if DL_PATH is None:
    raise FileNotFoundError(
        "Could not locate dataset_deep_learning.csv. Attach ${KAGGLE_INPUT_ROOT}/data_clean."
    )
if SPILL_PATH is None or SPILL_PATH.name != SPILL_FILENAME:
    raise FileNotFoundError(
        f"Refusing to run: could not locate required {SPILL_FILENAME}. "
        f"Upload revision/outputs/{SPILL_FILENAME} to Kaggle Input before running."
    )

actual_spill_sha256 = sha256_file(SPILL_PATH)
print(f"Using DL CSV    : {DL_PATH}")
print(f"Using SPILL CSV : {SPILL_PATH}")
print(f"SPILL SHA-256   : {actual_spill_sha256}")
assert actual_spill_sha256 == EXPECTED_SPILL_SHA256, (
    f"SHA-256 mismatch for {SPILL_PATH}! Expected {EXPECTED_SPILL_SHA256}, got {actual_spill_sha256}"
)
CONFIG_USED["data_provenance"]["dl_path"] = str(DL_PATH)
CONFIG_USED["data_provenance"]["spill_path"] = str(SPILL_PATH)
CONFIG_USED["data_provenance"]["actual_spill_sha256"] = actual_spill_sha256

if V6_HP_PATH is not None:
    with open(V6_HP_PATH, "r") as f_hp:
        loaded_hp = json.load(f_hp)
    if all(k in loaded_hp for k in ("TFT", "NBEATSx")):
        BEST_HP = loaded_hp
        CONFIG_USED["hyperparameters"] = BEST_HP
        print(f"Loaded exact BEST_HP from: {V6_HP_PATH}")

print(f"HAR v2 check    : {HAR1_PATH if HAR1_PATH else 'Not present (Hist-Exp computed directly from sample)'}")
print(f"RV60m CSV       : {RV60M_PATH if RV60M_PATH else 'Not attached (only required if RUN_3D=True)'}")

# ------------------------------------------------------------------
# 2. Exact Zero-ffill Merge, Float64 Targets & Warm-Up Discard (3b, 3c)
# ------------------------------------------------------------------
df_dl = pd.read_csv(DL_PATH)
df_spill = pd.read_csv(SPILL_PATH).sort_values("Date").reset_index(drop=True)
assert "net_spill_crypto_btc_idr_indodax" not in df_spill.columns, (
    "Target net spillover must stay excluded from SPILL CSV!"
)

# Inner join strictly on Date with ZERO forward-filling
df = (
    df_dl.merge(df_spill, on="Date", how="inner")
    .sort_values("Date")
    .reset_index(drop=True)
)
assert not df.isna().any().any(), "Unexpected NaNs after inner merge with spill_features_main_us1677_causal.csv!"
df["Date"] = df["Date"].astype(str)
df["real_date"] = df["Date"]
df["unique_id"] = "BTC_IDR"

# Keep target in strict float64, plus log(r^2 + 1e-10) for LogVar 3b
ret_btcidr_f64 = df["ret_crypto_btc_idr_indodax"].astype(np.float64)
df["y_r2"] = ret_btcidr_f64 ** 2
df["y_logr2"] = np.log(df["y_r2"].astype(np.float64) + 1e-10)

# Independent target recomputation check (3c)
indep_full_r2 = df_dl.set_index("Date").loc[df["Date"], "ret_crypto_btc_idr_indodax"].astype(np.float64).values ** 2
assert np.max(np.abs(df["y_r2"].values - indep_full_r2)) < 1e-12, "y_r2 diverges from independent ret^2!"

if "gap_days" not in df.columns:
    dt = pd.to_datetime(df["Date"])
    df["gap_days"] = dt.diff().dt.days.fillna(1).astype(int)
    df["is_monday"] = (dt.dt.dayofweek == 0).astype(int)
if "is_ihsg_holiday" not in df.columns:
    df["is_ihsg_holiday"] = 0

assert all(c in df.columns for c in SPILL_COLS), (
    f"Missing SPILL columns: {[c for c in SPILL_COLS if c not in df.columns]}"
)
assert set(SPILL_COLS) - set(NOSPILL_COLS) == set(SPILL_ONLY_9), (
    "SPILL vs NOSPILL feature sets do not differ in the exact 9 spillover columns!"
)
assert len(set(NOSPILL_COLS) - set(SPILL_COLS)) == 0

# Synthetic daily calendar index required by NeuralForecast
df["ds"] = pd.date_range("2020-01-01", periods=len(df), freq="D")

# Drop first WARMUP_DISCARD = 250 TVP-VAR burn-in rows while keeping exact val & test windows
df_post = df.iloc[WARMUP_DISCARD:].reset_index(drop=True)
n_test_rows = N_TEST_CUTOFFS + H - 1  # 335 + 10 - 1 = 344
n_fit = len(df_post) - n_test_rows
fit_df = df_post.iloc[:n_fit].reset_index(drop=True)
train_df = fit_df.iloc[: n_fit - N_VAL].reset_index(drop=True)
val_df = fit_df.iloc[n_fit - N_VAL :].reset_index(drop=True)
test_target_df = df_post.iloc[n_fit:].reset_index(drop=True)

# Strict assertions on sizes, exact date parity with df_spill post-warmup, and chronological ordering
assert len(df) == 1676, f"Expected merged df length 1676, got {len(df)}"
assert len(df_post) == 1426, f"Expected post-warmup length 1426, got {len(df_post)}"
assert (df_spill.iloc[WARMUP_DISCARD:]["Date"].astype(str).values == df_post["real_date"].values).all(), (
    "Loaded SPILL post-warm-up dates do not match df_post 1,426 dates identically!"
)
assert len(train_df) == 915, f"Expected train=915 after 250 warm-up drop, got {len(train_df)}"
assert len(val_df) == 167, f"Expected val=167, got {len(val_df)}"
assert len(test_target_df) == 344, f"Expected test_target_rows=344, got {len(test_target_df)}"
actual_origins = len(test_target_df) - H + 1
assert actual_origins == 335, f"Expected test origins=335, got {actual_origins}"
assert pd.to_datetime(val_df["real_date"].max()) < pd.to_datetime(test_target_df["real_date"].min()), (
    "Leakage check failed: validation dates overlap test target dates!"
)

# Check standardized exogenous features fit ONLY on train_df (915 rows) for 3b
EXOG_TRAIN_STATS = {}
fit_df_std = fit_df.copy()
df_post_std = df_post.copy()
for col in sorted(SPILL_COLS):
    vals_tr = train_df[col].astype(np.float64).values
    mu = float(np.mean(vals_tr))
    sd = float(np.std(vals_tr, ddof=0))
    if sd < 1e-12:
        sd = 1.0
    EXOG_TRAIN_STATS[col] = {"mean": mu, "std": sd}
    fit_df_std[col] = ((fit_df_std[col].astype(np.float64) - mu) / sd).astype(np.float64)
    df_post_std[col] = ((df_post_std[col].astype(np.float64) - mu) / sd).astype(np.float64)

TRAIN_MEAN_R2 = float(train_df["y_r2"].astype(np.float64).mean())
assert TRAIN_MEAN_R2 > 0.0 and np.isfinite(TRAIN_MEAN_R2)
FLOOR_SCALED_3B = float(VARIANCE_FLOOR_3B / TRAIN_MEAN_R2)
fit_df_std["y_r2_scaled"] = (fit_df_std["y_r2"].astype(np.float64) / TRAIN_MEAN_R2).astype(np.float64)
df_post_std["y_r2_scaled"] = (df_post_std["y_r2"].astype(np.float64) / TRAIN_MEAN_R2).astype(np.float64)
assert abs(float(fit_df_std.iloc[:len(train_df)]["y_r2_scaled"].mean()) - 1.0) < 1e-9

CONFIG_USED["data_provenance"]["train_exog_stats"] = EXOG_TRAIN_STATS
CONFIG_USED["data_provenance"]["train_mean_r2_915"] = TRAIN_MEAN_R2
CONFIG_USED["data_provenance"]["floor_scaled_3b"] = FLOOR_SCALED_3B
CONFIG_USED["splits"] = {
    "full_merged_rows": int(len(df)),
    "full_date_range": [str(df["real_date"].iloc[0]), str(df["real_date"].iloc[-1])],
    "post_warmup_rows": int(len(df_post)),
    "train_rows": int(len(train_df)),
    "train_date_range": [str(train_df["real_date"].iloc[0]), str(train_df["real_date"].iloc[-1])],
    "val_rows": int(len(val_df)),
    "val_date_range": [str(val_df["real_date"].iloc[0]), str(val_df["real_date"].iloc[-1])],
    "test_target_rows": int(len(test_target_df)),
    "test_target_date_range": [str(test_target_df["real_date"].iloc[0]), str(test_target_df["real_date"].iloc[-1])],
    "test_origins": int(actual_origins),
}

print("\n=== SAMPLE & ALIGNMENT SUMMARY ===")
print(f"Merged Dataset       : shape={df.shape}, dates={df['real_date'].iloc[0]} -> {df['real_date'].iloc[-1]} (0 ffill)")
print(f"Post-Warmup (-250)   : shape={df_post.shape}, dates={df_post['real_date'].iloc[0]} -> {df_post['real_date'].iloc[-1]}")
print(f"Train Split          : {len(train_df)} rows ({train_df['real_date'].iloc[0]} -> {train_df['real_date'].iloc[-1]}), mean_r2={TRAIN_MEAN_R2:.6e}")
print(f"Val Split            : {len(val_df)} rows ({val_df['real_date'].iloc[0]} -> {val_df['real_date'].iloc[-1]})")
print(f"Test Targets         : {len(test_target_df)} rows / {actual_origins} origins")
print(f"First Target Date    : {test_target_df['real_date'].iloc[0]} (h=1 of origin 0)")
print(f"Last Target Date     : {test_target_df['real_date'].iloc[-1]} (h=10 of origin {actual_origins-1})")

# ------------------------------------------------------------------
# 3. Alignment Manifest & Causal Hist-Exp Benchmark
# ------------------------------------------------------------------
manifest_rows = []
base_fit_end = len(fit_df)  # 1082 inside df_post
for o_idx in range(N_TEST_CUTOFFS):
    cutoff_pos_post = base_fit_end - 1 + o_idx
    cutoff_dt = str(df_post.loc[cutoff_pos_post, "real_date"])
    row = {
        "origin_idx": int(o_idx),
        "cutoff_date": cutoff_dt,
        "train_start_idx": 0,
        "train_end_idx": int(len(train_df) - 1),
        "val_start_idx": int(len(train_df)),
        "val_end_idx": int(len(fit_df) - 1),
        "train_start_idx_full": int(WARMUP_DISCARD),
        "train_end_idx_full": int(WARMUP_DISCARD + len(train_df) - 1),
        "val_start_idx_full": int(WARMUP_DISCARD + len(train_df)),
        "val_end_idx_full": int(WARMUP_DISCARD + len(fit_df) - 1),
        "train_start_date": str(train_df["real_date"].iloc[0]),
        "train_end_date": str(train_df["real_date"].iloc[-1]),
        "val_start_date": str(val_df["real_date"].iloc[0]),
        "val_end_date": str(val_df["real_date"].iloc[-1]),
    }
    for h_step in range(1, H + 1):
        row[f"target_date_h{h_step}"] = str(df_post.loc[cutoff_pos_post + h_step, "real_date"])
    manifest_rows.append(row)

ALIGNMENT_MANIFEST = pd.DataFrame(manifest_rows)
MANIFEST_PATH = OUTPUT_DIR / "alignment_manifest.csv"
ALIGNMENT_MANIFEST.to_csv(MANIFEST_PATH, index=False)

assert ALIGNMENT_MANIFEST.loc[0, "cutoff_date"] == "2025-04-23"
assert ALIGNMENT_MANIFEST.loc[0, "target_date_h1"] == "2025-04-24"
assert ALIGNMENT_MANIFEST.loc[N_TEST_CUTOFFS - 1, "target_date_h10"] == "2026-09-04"

# Expanding Hist-Exp aligned by origin_idx (matches har_preds_T0_h1_v2 Hist_Exp to float precision)
df_dl_sorted = df_dl.sort_values("Date").reset_index(drop=True)
df_dl_sorted["y_r2"] = df_dl_sorted["ret_crypto_btc_idr_indodax"].astype(np.float64) ** 2
HIST_EXP_BY_ORIGIN = np.array(
    [df_dl_sorted.loc[df_dl_sorted["Date"] <= ALIGNMENT_MANIFEST.loc[o, "cutoff_date"], "y_r2"].mean() for o in range(N_TEST_CUTOFFS)],
    dtype=float,
)

if HAR1_PATH is not None:
    df_har1 = pd.read_csv(HAR1_PATH).sort_values("Date").reset_index(drop=True)
    assert len(df_har1) == N_TEST_CUTOFFS
    assert (df_har1["Date"].astype(str).values == ALIGNMENT_MANIFEST["target_date_h1"].values).all()
    hexp_diff = float(np.max(np.abs(HIST_EXP_BY_ORIGIN - df_har1["Hist_Exp"].values)))
    print(f"HAR v2 h=1 cross-check PASSED (max |Hist-Exp diff| = {hexp_diff:.2e})")

DS_TO_DATE = dict(zip(pd.to_datetime(df_post["ds"]), df_post["real_date"].astype(str)))
CUTOFF_DS_TO_ORIGIN = {
    pd.Timestamp(df_post.loc[base_fit_end - 1 + o_idx, "ds"]): o_idx
    for o_idx in range(N_TEST_CUTOFFS)
}
DATE_TO_R2 = dict(zip(df_post["real_date"].astype(str), df_post["y_r2"].astype(np.float64)))
DATE_TO_LOGR2 = dict(zip(df_post["real_date"].astype(str), df_post["y_logr2"].astype(np.float64)))
print(f"Saved alignment_manifest.csv ({len(ALIGNMENT_MANIFEST)} origins): {MANIFEST_PATH}")


## Section 1: Data Loading, SHA-256 Provenance Guard, Alignment Assertions & Shared Helpers

Enforces Round-4 safeguards:
- **`3a` Loader Fix**: Loads `spill_features_main_us1677_causal.csv` only, verifies its exact SHA-256 checksum, aborts if missing or if any `tvp_var_dynamic_features*.csv` file is referenced, and performs an exact date inner merge with **zero `.ffill()`**.
- **`3b` Split & Feature Checks**: Verifies exact `1,426` post-warm-up date parity (`train=915`, `val=167`, `test=335` origins / `344` rows), zero NaNs, and exact `9`-column `SPILL` vs `NOSPILL` separation.
- **`3c` Float64 Precision Check**: Independently computes realized `r^2` in `float64` (`max_abs_diff < 1e-12`) and validates NeuralForecast `float32` `nf_y` alignment (`rtol=1e-6`).
- **`3e` Collapse Guard**: Defines `check_prediction_health` to assert `std(y_pred) > 1e-12` and log `<= 0` prediction shares after every fit.


In [ ]:
# ------------------------------------------------------------------
# Shared v6 Model Factory, SoftplusScaledNBEATSx & Verification Guards
# ------------------------------------------------------------------
class SoftplusScaledNBEATSx(NBEATSx):
    '''
    N-BEATSx preserving the exact [256, 256] x 3 block architecture while replacing
    unconstrained output projection with:
      1) output Linear(h, h) initialized with weight=0.1*I and bias=log(exp(1.0)-1)
         (so initial Softplus prediction ~ 1.0 in train_mean_r2=1.0 scaled units),
      2) F.softplus(...) + floor_scaled where floor_scaled = 1e-8 / train_mean_r2.
    Requires target scaled by (r^2 / train_mean_r2), standardized exog, and scaler_type="identity".
    '''
    def __init__(self, *args, floor_scaled: float = 1e-5, init_bias_val: float = math.log(math.exp(1.0) - 1.0), **kwargs):
        super().__init__(*args, **kwargs)
        self.floor_scaled = float(floor_scaled)
        self.init_bias_val = float(init_bias_val)
        self.output_head = nn.Linear(self.h, self.h)
        with torch.no_grad():
            self.output_head.weight.copy_(torch.eye(self.h) * 0.1)
            self.output_head.bias.fill_(self.init_bias_val)

    def forward(self, windows_batch):
        insample_y = windows_batch["insample_y"].squeeze(-1)
        insample_mask = windows_batch["insample_mask"].squeeze(-1)
        futr_exog = windows_batch["futr_exog"]
        hist_exog = windows_batch["hist_exog"]
        stat_exog = windows_batch["stat_exog"]

        residuals = insample_y.flip(dims=(-1,))
        insample_mask = insample_mask.flip(dims=(-1,))

        block_sum = torch.zeros((len(insample_y), self.h, 1), device=insample_y.device, dtype=insample_y.dtype)
        for block in self.blocks:
            backcast, block_forecast = block(
                insample_y=residuals,
                futr_exog=futr_exog,
                hist_exog=hist_exog,
                stat_exog=stat_exog,
            )
            residuals = (residuals - backcast) * insample_mask
            block_sum = block_sum + block_forecast

        raw = self.output_head(block_sum.squeeze(-1)).unsqueeze(-1)
        return F.softplus(raw) + self.floor_scaled


def get_weights_checksum_from_dict(state_dict) -> str:
    h = hashlib.md5()
    for k, v in sorted(state_dict.items()):
        h.update(k.encode("utf-8"))
        h.update(v.detach().cpu().numpy().tobytes())
    return h.hexdigest()


def get_weights_checksum(model) -> str:
    return get_weights_checksum_from_dict(model.state_dict())


class RestoreBestWeightsCallback(pl.Callback):
    '''In-memory best validation checkpoint restorer matching 06_dl_v6_multiseed.'''
    def __init__(self):
        super().__init__()
        self.best_score = None
        self.best_step = None
        self.best_state_dict = None
        self.best_checksum = None
        self.actual_final_step = 0
        self.stopped_epoch = 0
        self.es_triggered = False
        self.restored = False
        self.history = []

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        self.actual_final_step = trainer.global_step

    def on_validation_end(self, trainer, pl_module):
        if getattr(trainer, "sanity_checking", False) or trainer.global_step == 0:
            return
        val_loss = trainer.callback_metrics.get("ptl/val_loss", None)
        if val_loss is not None:
            v_val = float(val_loss)
            self.history.append((int(trainer.global_step), v_val))
            if self.best_score is None or v_val < self.best_score:
                self.best_score = v_val
                self.best_step = int(trainer.global_step)
                self.best_state_dict = {
                    k: v.detach().cpu().clone() for k, v in pl_module.state_dict().items()
                }
                self.best_checksum = get_weights_checksum_from_dict(self.best_state_dict)

    def on_train_end(self, trainer, pl_module):
        self.actual_final_step = int(trainer.global_step)
        for cb in trainer.callbacks:
            if isinstance(cb, EarlyStopping):
                self.stopped_epoch = int(cb.stopped_epoch)
                self.es_triggered = (cb.stopped_epoch > 0)
        if self.best_state_dict is not None:
            pl_module.load_state_dict(self.best_state_dict)
            self.restored = True


def build_v6_model(
    model_name: str,
    fset_cols: list,
    seed: int,
    h: int = H,
    loss_override=None,
    scaler_type: str = "standard",
    use_softplus_scaled: bool = False,
    floor_scaled: float = 1e-5,
):
    '''Instantiate TFT, NBEATSx, or SoftplusScaledNBEATSx with explicit device=1 config.'''
    hp = BEST_HP[model_name]
    loss_inst = loss_override if loss_override is not None else (
        MAE() if hp["loss_name"] == "MAE" else MSE()
    )
    tracker = RestoreBestWeightsCallback()

    kwargs = {
        "h": h,
        "input_size": hp["input_size"],
        "hist_exog_list": fset_cols,
        "max_steps": MAX_STEPS,
        "val_check_steps": VAL_CHECK_STEPS,
        "early_stop_patience_steps": PATIENCE,
        "loss": loss_inst,
        "random_seed": seed,
        "scaler_type": scaler_type,
        "learning_rate": hp["learning_rate"],
    }
    if model_name == "TFT":
        kwargs["hidden_size"] = hp["hidden_size"]
        model = TFT(**kwargs)
    elif model_name == "NBEATSx":
        kwargs["mlp_units"] = hp["mlp_units"]
        if use_softplus_scaled:
            model = SoftplusScaledNBEATSx(**kwargs, floor_scaled=floor_scaled)
        else:
            model = NBEATSx(**kwargs)
    else:
        raise ValueError(f"Unsupported model: {model_name}")

    # Explicit single-device configuration (3d)
    device_cfg = {"accelerator": "cuda", "devices": 1} if torch.cuda.is_available() else {"accelerator": "cpu", "devices": 1}
    model.trainer_kwargs.update({
        "enable_progress_bar": bool(SHOW_TRAINER_STEP_BAR),
        "enable_checkpointing": False,
        "logger": False,
        **device_cfg,
    })

    if "callbacks" not in model.trainer_kwargs or model.trainer_kwargs["callbacks"] is None:
        model.trainer_kwargs["callbacks"] = []
    model.trainer_kwargs["callbacks"].append(tracker)
    assert any(isinstance(cb, EarlyStopping) for cb in model.trainer_kwargs["callbacks"])
    return model, tracker


def prepare_nf_frame(src_df: pd.DataFrame, target_col: str, fcols: list, target_scale: float = 1.0) -> pd.DataFrame:
    out = pd.DataFrame({
        "unique_id": src_df["unique_id"].values,
        "ds": pd.to_datetime(src_df["ds"].values),
        "y": (src_df[target_col].values.astype(np.float64) * target_scale).astype("float32"),
    })
    for col in sorted(fcols):
        out[col] = src_df[col].values.astype("float32")
    return out


def verify_target_precision_3c(label: str, y_true_f64: np.ndarray, target_dates: np.ndarray, nf_y_vals: np.ndarray, nf_expected_f64: np.ndarray, ref_map: dict):
    '''3c: Verify float64 y_true vs independent r^2 (<1e-12) and float32 nf_y parity (rtol=1e-6).'''
    indep = np.array([ref_map[str(d)] for d in target_dates], dtype=np.float64)
    diff_f64 = float(np.max(np.abs(y_true_f64 - indep)))
    assert diff_f64 < 1e-12, f"Float64 target mismatch on {label}: {diff_f64:.2e}"
    f32_ok = bool(np.allclose(nf_y_vals, np.float32(nf_expected_f64), rtol=1e-6, atol=1e-12))
    max_f32_rel = float(np.max(np.abs(nf_y_vals - nf_expected_f64) / np.maximum(np.abs(nf_expected_f64), 1e-12)))
    assert f32_ok, f"NeuralForecast float32 target check failed on {label} (max_rel={max_f32_rel:.2e})!"
    CONFIG_USED["target_precision_checks"].append({
        "run": label,
        "max_abs_diff_f64_vs_indep": diff_f64,
        "nf_y_allclose_f32_rtol1e6": f32_ok,
        "max_rel_diff_nf_y_vs_f64": max_f32_rel,
    })


def check_prediction_health_3e(y_pred: np.ndarray, label: str, fit_log_entry: dict):
    '''3e: Guard against constant/collapsed outputs (std <= 1e-12) and log <= 0 prediction share.'''
    yp = np.asarray(y_pred, dtype=np.float64)
    p_std = float(np.std(yp))
    p_min = float(np.min(yp))
    p_max = float(np.max(yp))
    p_mean = float(np.mean(yp))
    neg_sh = float(np.mean(yp < 0.0))
    le0_sh = float(np.mean(yp <= 0.0))
    collapsed = bool((not np.isfinite(p_std)) or (p_std <= 1e-12))

    fit_log_entry.update({
        "pred_std": p_std,
        "pred_min": p_min,
        "pred_max": p_max,
        "pred_mean": p_mean,
        "neg_share": neg_sh,
        "le_zero_share": le0_sh,
        "collapsed": collapsed,
    })
    if collapsed:
        msg = f"COLLAPSE DETECTED in {label}: std={p_std:.3e}, min={p_min:.3e}, max={p_max:.3e}, le0={le0_sh*100:.2f}%"
        CONFIG_USED["collapse_warnings"].append({"run": label, **fit_log_entry})
        print(f"!!! ALERT: {msg} !!!")
        raise RuntimeError(msg)


def align_and_verify_cv(
    cv_raw: pd.DataFrame,
    model_col: str,
    model_name: str,
    feature_set: str,
    variant: str,
    seed: int,
    date_to_true: dict = None,
    expected_origins=None,
) -> pd.DataFrame:
    if date_to_true is None:
        date_to_true = DATE_TO_R2

    cv = cv_raw.copy().reset_index(drop=True)
    cv["ds"] = pd.to_datetime(cv["ds"])
    cv["cutoff"] = pd.to_datetime(cv["cutoff"])
    cv["origin_idx"] = cv["cutoff"].map(CUTOFF_DS_TO_ORIGIN)
    assert not cv["origin_idx"].isna().any(), "Unmapped cutoff timestamp in CV output!"
    cv["origin_idx"] = cv["origin_idx"].astype(int)

    cv["cutoff_date"] = cv["cutoff"].map(DS_TO_DATE)
    cv["target_date"] = cv["ds"].map(DS_TO_DATE)
    assert not cv["cutoff_date"].isna().any(), "Unmapped cutoff_date in CV output!"
    assert not cv["target_date"].isna().any(), "Unmapped target_date in CV output!"

    cv = cv.sort_values(["origin_idx", "ds"]).reset_index(drop=True)
    cv["horizon"] = cv.groupby("origin_idx").cumcount() + 1
    assert set(cv["horizon"].unique()) == set(range(1, H + 1)), "Missing horizons in CV output!"

    if expected_origins is not None:
        got_origins = sorted(cv["origin_idx"].unique().tolist())
        exp_list = list(expected_origins)
        assert got_origins == exp_list, (
            f"Origin mismatch: got {got_origins[:3]}..{got_origins[-2:]} vs expected {exp_list[:3]}..{exp_list[-2:]}"
        )

    expected_target = [
        ALIGNMENT_MANIFEST.loc[o, f"target_date_h{h}"]
        for o, h in zip(cv["origin_idx"].values, cv["horizon"].values)
    ]
    expected_cutoff = [ALIGNMENT_MANIFEST.loc[o, "cutoff_date"] for o in cv["origin_idx"].values]
    assert (cv["target_date"].values == np.array(expected_target)).all(), "target_date shift error!"
    assert (cv["cutoff_date"].values == np.array(expected_cutoff)).all(), "cutoff_date shift error!"
    assert (pd.to_datetime(cv["target_date"]) > pd.to_datetime(cv["cutoff_date"])).all(), "target_date <= cutoff_date!"

    y_true_exact = cv["target_date"].map(date_to_true).astype(np.float64)
    assert not y_true_exact.isna().any(), "Missing y_true for target_date!"

    return pd.DataFrame({
        "model": model_name,
        "feature_set": feature_set,
        "variant": variant,
        "seed": int(seed),
        "origin_idx": cv["origin_idx"].values.astype(int),
        "cutoff_date": cv["cutoff_date"].values,
        "target_date": cv["target_date"].values,
        "horizon": cv["horizon"].values.astype(int),
        "y_true": y_true_exact.values,
        "y_pred": cv[model_col].values.astype(np.float64),
        "nf_y": cv["y"].values.astype(np.float64),
    })


def fit_and_cv_aligned(
    model_name: str,
    fcols: list,
    fit_src: pd.DataFrame,
    all_src: pd.DataFrame,
    target_col: str,
    seed: int,
    feature_set: str,
    variant: str,
    section_tag: str,
    n_windows: int = N_TEST_CUTOFFS,
    val_size: int = N_VAL,
    target_scale: float = 1.0,
    loss_override=None,
    scaler_type: str = "standard",
    use_softplus_scaled: bool = False,
    floor_scaled: float = 1e-5,
    date_to_true: dict = None,
    expected_origins=None,
):
    set_all_seeds(seed)
    t_run = time.time()
    fit_nf = prepare_nf_frame(fit_src, target_col, fcols, target_scale=target_scale)
    all_nf = prepare_nf_frame(all_src, target_col, fcols, target_scale=target_scale)

    model, cb = build_v6_model(
        model_name=model_name,
        fset_cols=fcols,
        seed=seed,
        h=H,
        loss_override=loss_override,
        scaler_type=scaler_type,
        use_softplus_scaled=use_softplus_scaled,
        floor_scaled=floor_scaled,
    )
    model_col_out = model.__class__.__name__
    nf = NeuralForecast(models=[model], freq="D")
    nf.fit(df=fit_nf, val_size=val_size)

    m_fitted = nf.models[0]
    chk_post_fit = get_weights_checksum(m_fitted)
    if cb.best_checksum is not None:
        assert chk_post_fit == cb.best_checksum, "Restored weights checksum mismatch!"
        assert cb.best_step > 0, f"Invalid best_step={cb.best_step}"

    cv_raw = nf.cross_validation(
        df=all_nf,
        n_windows=n_windows,
        step_size=1,
        use_fitted=True,
    ).reset_index(drop=True)

    chk_post_cv = get_weights_checksum(m_fitted)
    assert chk_post_cv == chk_post_fit, "Model weights mutated during cross_validation!"

    aligned = align_and_verify_cv(
        cv_raw=cv_raw,
        model_col=model_col_out,
        model_name=model_name,
        feature_set=feature_set,
        variant=variant,
        seed=seed,
        date_to_true=date_to_true,
        expected_origins=expected_origins,
    )

    elapsed = round(time.time() - t_run, 2)
    fit_entry = {
        "section": section_tag,
        "model": model_name,
        "feature_set": feature_set,
        "variant": variant,
        "seed": int(seed),
        "best_step": cb.best_step,
        "best_val_loss": cb.best_score,
        "actual_final_step": cb.actual_final_step,
        "es_triggered": cb.es_triggered,
        "weights_restored": cb.restored,
        "fit_cv_seconds": elapsed,
    }
    CONFIG_USED["fit_logs"].append(fit_entry)
    return aligned, nf, m_fitted, elapsed, fit_entry

print("Shared v6 model factory, SoftplusScaledNBEATSx, and Round-4 verification helpers ready.")


## Section 2 (Step 3a): Warm-Up-Excluded Retraining of `TFT` & `NBEATSx` (20 Runs)

Retrains `TFT` (`MSE`, `input_size=50`, `hidden_size=64`, `lr=0.001`) and `NBEATSx` (`MAE`, `input_size=20`, `lr=0.0003`, `mlp_units=[[256,256]]*3`) across `SPILL` (from verified `spill_features_main_us1677_causal.csv`) and `NOSPILL` over `5` seeds `[42, 7, 13, 21, 99]` on `train=915, val=167, test=335`. Runs `3c` target precision checks and `3e` collapse guard on every fit.


In [ ]:
# ------------------------------------------------------------------
# Section 2 (3a): Retrain TFT & NBEATSx Excluding First 250 Warm-Up Rows
# ------------------------------------------------------------------
if RUN_3A:
    t0_3a = time.time()
    print("=== SECTION 2 (3a): WARM-UP EXCLUDED RETRAINING (20 RUNS) ===")
    try:
        configs_3a = [("TFT", "SPILL"), ("TFT", "NOSPILL"), ("NBEATSx", "SPILL"), ("NBEATSx", "NOSPILL")]
        runs_3a = [(mname, fset, seed) for mname, fset in configs_3a for seed in SEEDS]
        with tqdm(runs_3a, desc="Step 3a (NoWarmup250)", unit="run", disable=not SHOW_PROGRESS_BAR) as pbar:
            for mname, fset, seed in pbar:
                pbar.set_postfix_str(f"{mname}-{fset} s={seed}")
                fcols = FEATURE_SETS[fset]
                aligned, nf_obj, _, sec, fit_entry = fit_and_cv_aligned(
                    model_name=mname,
                    fcols=fcols,
                    fit_src=fit_df,
                    all_src=df_post,
                    target_col="y_r2",
                    seed=seed,
                    feature_set=fset,
                    variant="NoWarmup250",
                    section_tag="3a",
                    n_windows=N_TEST_CUTOFFS,
                    val_size=N_VAL,
                    scaler_type="standard",
                    date_to_true=DATE_TO_R2,
                    expected_origins=range(N_TEST_CUTOFFS),
                )
                label = f"3a:{mname}-{fset}:s{seed}"
                verify_target_precision_3c(
                    label=label,
                    y_true_f64=aligned["y_true"].values,
                    target_dates=aligned["target_date"].values,
                    nf_y_vals=aligned["nf_y"].values,
                    nf_expected_f64=aligned["y_true"].values,
                    ref_map=DATE_TO_R2,
                )
                check_prediction_health_3e(aligned["y_pred"].values, label=label, fit_log_entry=fit_entry)

                out_df = aligned.drop(columns=["nf_y"])
                per_seed_file = PER_SEED_DIR / f"3a_{mname}_{fset}_NoWarmup250_s{seed}.csv"
                out_df.to_csv(per_seed_file, index=False)
                LONG_FRAMES.append(out_df)

                neg_rate = float(np.mean(out_df["y_pred"].values < 0.0))
                le0_rate = float(np.mean(out_df["y_pred"].values <= 0.0))
                tqdm.write(
                    f"  [3a] {mname}-{fset} seed={seed:<2} | time={sec:5.1f}s | "
                    f"std={fit_entry['pred_std']:.2e} | <=0={le0_rate*100:5.2f}% | saved={per_seed_file.name}"
                )
                del nf_obj
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

        CONFIG_USED["timings_sec"]["3a"] = round(time.time() - t0_3a, 2)
        print(f"Section 3a finished in {CONFIG_USED['timings_sec']['3a']/60:.2f} min.")
    except Exception as exc:
        CONFIG_USED["timings_sec"]["3a"] = round(time.time() - t0_3a, 2)
        CONFIG_USED["errors"]["3a"] = traceback.format_exc()
        print(f"ERROR in Section 3a (logged to config_used.json):\n{exc}")
else:
    print("Section 3a skipped (RUN_3A=False).")


## Section 3 (Step 3b Redesigned): `LogVar` & `SoftplusScaled` Non-Negative `NBEATSx` (20 Fits)

Evaluates two strictly positive volatility parameterizations across `5` seeds $\times$ `2` feature sets (`20` independent model fits) using exogenous inputs standardized strictly on `train=915`:
1. **Variant V-A (`LogVar`, 10 fits)**: Models $z_t = \log(r_t^2 + 10^{-10})$ with `scaler_type="standard"`, `MAE()` loss, and standardized exogenous features fit on `train=915` only. Inverts log predictions via Duan smearing $\hat{\sigma}^2_{t+h} = \exp(\hat{z}_{t+h}) \cdot \widehat{d}_{\text{val}}$, where $\widehat{d}_{\text{val}} = \text{mean}(\exp(z_{\text{val}} - \hat{z}_{\text{val}}))$ is estimated strictly on the `158` causal validation windows inside `fit_df` (**zero test data**).
2. **Variant V-B (`SoftplusScaled`, 10 fits)**: Models scaled variance $y_t / \bar{y}_{\text{train}}$ (target mean $= 1.0$ on `train=915`) with `scaler_type="identity"`, train-standardized exogenous features, `Softplus` output head, additive variance floor `1e-8` (strictly `> 0`), and output linear bias initialized to $\log(e^{1.0} - 1) \approx 0.541325$ so initial predictions start near `train_mean_r2`.

Both variants run `3c` precision verification and `3e` collapse assertions (`std(y_pred) > 1e-12`, `<= 0` share `== 0`).


In [ ]:
# ------------------------------------------------------------------
# Section 3 (3b Redesigned): LogVar (10 fits) & SoftplusScaled (10 fits)
# ------------------------------------------------------------------
if RUN_3B:
    t0_3b = time.time()
    print("=== SECTION 3 (3b REDESIGNED): LogVar & SoftplusScaled N-BEATSx (20 FITS) ===")
    try:
        logvar_diagnostics = []
        softplus_diagnostics = []
        runs_3b = [
            (variant_name, fset, seed)
            for variant_name in ["LogVar", "SoftplusScaled"]
            for fset in ["SPILL", "NOSPILL"]
            for seed in SEEDS
        ]

        with tqdm(runs_3b, desc="Step 3b (LogVar & SoftplusScaled)", unit="fit", disable=not SHOW_PROGRESS_BAR) as pbar_3b:
            for variant_name, fset, seed in pbar_3b:
                pbar_3b.set_postfix_str(f"{variant_name}-{fset} s={seed}")
                fcols = FEATURE_SETS[fset]

                if variant_name == "LogVar":
                    aligned_log, nf_obj, _, sec, fit_entry = fit_and_cv_aligned(
                        model_name="NBEATSx",
                        fcols=fcols,
                        fit_src=fit_df_std,
                        all_src=df_post_std,
                        target_col="y_logr2",
                        seed=seed,
                        feature_set=fset,
                        variant="LogVar",
                        section_tag="3b",
                        n_windows=N_TEST_CUTOFFS,
                        val_size=N_VAL,
                        target_scale=1.0,
                        loss_override=None,
                        scaler_type="standard",
                        date_to_true=DATE_TO_R2,
                        expected_origins=range(N_TEST_CUTOFFS),
                    )
                    label = f"3b:LogVar-{fset}:s{seed}"
                    # Check target precision on both log(r^2+1e-10) in NF and float64 r^2 y_true
                    expected_log_arr = np.array([DATE_TO_LOGR2[str(d)] for d in aligned_log["target_date"].values], dtype=np.float64)
                    verify_target_precision_3c(
                        label=label,
                        y_true_f64=aligned_log["y_true"].values,
                        target_dates=aligned_log["target_date"].values,
                        nf_y_vals=aligned_log["nf_y"].values,
                        nf_expected_f64=expected_log_arr,
                        ref_map=DATE_TO_R2,
                    )

                    # Strict causal validation Duan smearing over the 158 validation windows inside fit_df_std (zero test data!)
                    fit_nf_log = prepare_nf_frame(fit_df_std, "y_logr2", fcols, target_scale=1.0)
                    n_val_windows = N_VAL - H + 1  # 167 - 10 + 1 = 158
                    cv_val_log = nf_obj.cross_validation(
                        df=fit_nf_log,
                        n_windows=n_val_windows,
                        step_size=1,
                        use_fitted=True,
                    ).reset_index(drop=True)
                    cv_val_log["ds"] = pd.to_datetime(cv_val_log["ds"])
                    cv_val_log["cutoff"] = pd.to_datetime(cv_val_log["cutoff"])
                    cv_val_log = cv_val_log.sort_values(["cutoff", "ds"]).reset_index(drop=True)
                    cv_val_log["horizon"] = cv_val_log.groupby("cutoff").cumcount() + 1
                    assert len(cv_val_log) == n_val_windows * H, (
                        f"Expected {n_val_windows * H} val CV rows, got {len(cv_val_log)}"
                    )
                    val_max_ds = pd.to_datetime(val_df["ds"].max())
                    assert cv_val_log["ds"].max() <= val_max_ds, "Duan smearing validation leakage detected!"

                    val_log_resid = cv_val_log["y"].values.astype(np.float64) - cv_val_log["NBEATSx"].values.astype(np.float64)
                    duan_smearing = float(np.mean(np.exp(np.clip(val_log_resid, -50.0, 20.0))))
                    assert np.isfinite(duan_smearing) and duan_smearing > 0.0, f"Invalid Duan smearing {duan_smearing}"

                    duan_by_h = {}
                    for hv in range(1, H + 1):
                        sub_h = cv_val_log[cv_val_log["horizon"] == hv]
                        res_h = sub_h["y"].values.astype(np.float64) - sub_h["NBEATSx"].values.astype(np.float64)
                        duan_by_h[str(hv)] = float(np.mean(np.exp(np.clip(res_h, -50.0, 20.0))))

                    raw_log_pred = aligned_log["y_pred"].values.astype(np.float64)
                    pred_var = np.exp(np.clip(raw_log_pred, -50.0, 20.0)) * duan_smearing
                    check_prediction_health_3e(pred_var, label=label, fit_log_entry=fit_entry)

                    logvar_diagnostics.append({
                        "feature_set": fset,
                        "seed": int(seed),
                        "duan_smearing_val158": duan_smearing,
                        "duan_h1": duan_by_h["1"],
                        "duan_h10": duan_by_h["10"],
                        "pred_var_min": float(pred_var.min()),
                        "pred_var_mean": float(pred_var.mean()),
                        "pred_var_std": float(pred_var.std()),
                    })

                    out_lv = aligned_log.drop(columns=["nf_y"]).copy()
                    out_lv["y_pred"] = pred_var
                    ps_file = PER_SEED_DIR / f"3b_NBEATSx_{fset}_LogVar_s{seed}.csv"
                    out_lv.assign(raw_log_pred=raw_log_pred, duan_smearing=duan_smearing).to_csv(ps_file, index=False)
                    LONG_FRAMES.append(out_lv)

                    for hv in range(1, H + 1):
                        hmask = out_lv["horizon"].values == hv
                        yv = out_lv.loc[hmask, "y_pred"].values
                        NEG_REPORT_ROWS.append({
                            "model": "NBEATSx",
                            "feature_set": fset,
                            "variant": "LogVar",
                            "seed": int(seed),
                            "horizon": int(hv),
                            "n_obs": int(hmask.sum()),
                            "neg_share_before_clip": float(np.mean(yv < 0.0)),
                            "le_zero_share_before_clip": float(np.mean(yv <= 0.0)),
                            "neg_share_after_clip": float(np.mean(yv < 0.0)),
                            "le_zero_share_after_clip": float(np.mean(yv <= 0.0)),
                        })

                    tqdm.write(
                        f"  [3b] LogVar-{fset:<7} s={seed:<2} | time={sec:4.1f}s | "
                        f"duan={duan_smearing:.3f} | std={pred_var.std():.2e} | min={pred_var.min():.2e} | saved={ps_file.name}"
                    )

                else:
                    aligned_sp, nf_obj, _, sec, fit_entry = fit_and_cv_aligned(
                        model_name="NBEATSx",
                        fcols=fcols,
                        fit_src=fit_df_std,
                        all_src=df_post_std,
                        target_col="y_r2_scaled",
                        seed=seed,
                        feature_set=fset,
                        variant="SoftplusScaled",
                        section_tag="3b",
                        n_windows=N_TEST_CUTOFFS,
                        val_size=N_VAL,
                        target_scale=1.0,
                        loss_override=MAE(),
                        scaler_type="identity",
                        use_softplus_scaled=True,
                        floor_scaled=FLOOR_SCALED_3B,
                        date_to_true=DATE_TO_R2,
                        expected_origins=range(N_TEST_CUTOFFS),
                    )
                    label = f"3b:SoftplusScaled-{fset}:s{seed}"
                    expected_scaled_arr = aligned_sp["y_true"].values.astype(np.float64) / TRAIN_MEAN_R2
                    verify_target_precision_3c(
                        label=label,
                        y_true_f64=aligned_sp["y_true"].values,
                        target_dates=aligned_sp["target_date"].values,
                        nf_y_vals=aligned_sp["nf_y"].values,
                        nf_expected_f64=expected_scaled_arr,
                        ref_map=DATE_TO_R2,
                    )

                    pred_var = aligned_sp["y_pred"].values.astype(np.float64) * TRAIN_MEAN_R2
                    assert np.all(pred_var >= VARIANCE_FLOOR_3B * (1.0 - 1e-6)), (
                        f"SoftplusScaled violated variance floor 1e-8: min={pred_var.min():.3e}"
                    )
                    check_prediction_health_3e(pred_var, label=label, fit_log_entry=fit_entry)

                    softplus_diagnostics.append({
                        "feature_set": fset,
                        "seed": int(seed),
                        "train_mean_r2": TRAIN_MEAN_R2,
                        "floor_scaled": FLOOR_SCALED_3B,
                        "pred_var_min": float(pred_var.min()),
                        "pred_var_mean": float(pred_var.mean()),
                        "pred_var_std": float(pred_var.std()),
                    })

                    out_sp = aligned_sp.drop(columns=["nf_y"]).copy()
                    out_sp["y_pred"] = pred_var
                    ps_file = PER_SEED_DIR / f"3b_NBEATSx_{fset}_SoftplusScaled_s{seed}.csv"
                    out_sp.to_csv(ps_file, index=False)
                    LONG_FRAMES.append(out_sp)

                    for hv in range(1, H + 1):
                        hmask = out_sp["horizon"].values == hv
                        yv = out_sp.loc[hmask, "y_pred"].values
                        NEG_REPORT_ROWS.append({
                            "model": "NBEATSx",
                            "feature_set": fset,
                            "variant": "SoftplusScaled",
                            "seed": int(seed),
                            "horizon": int(hv),
                            "n_obs": int(hmask.sum()),
                            "neg_share_before_clip": float(np.mean(yv < 0.0)),
                            "le_zero_share_before_clip": float(np.mean(yv <= 0.0)),
                            "neg_share_after_clip": float(np.mean(yv < 0.0)),
                            "le_zero_share_after_clip": float(np.mean(yv <= 0.0)),
                        })

                    tqdm.write(
                        f"  [3b] SoftplusScaled-{fset:<7} s={seed:<2} | time={sec:4.1f}s | "
                        f"std={pred_var.std():.2e} | min={pred_var.min():.2e} | saved={ps_file.name}"
                    )

                del nf_obj
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

        df_neg_seed = pd.DataFrame(NEG_REPORT_ROWS)
        df_neg_report = (
            df_neg_seed.groupby(["model", "feature_set", "variant", "horizon"], as_index=False)
            .agg(
                n_obs=("n_obs", "sum"),
                n_seeds=("seed", "nunique"),
                neg_share_before_clip=("neg_share_before_clip", "mean"),
                le_zero_share_before_clip=("le_zero_share_before_clip", "mean"),
                neg_share_after_clip=("neg_share_after_clip", "mean"),
                le_zero_share_after_clip=("le_zero_share_after_clip", "mean"),
            )
            .sort_values(["variant", "feature_set", "horizon"])
            .reset_index(drop=True)
        )
        NEG_REPORT_PATH = OUTPUT_DIR / "negative_forecast_report.csv"
        df_neg_report.to_csv(NEG_REPORT_PATH, index=False)

        CONFIG_USED["logvar_diagnostics_3b"] = logvar_diagnostics
        CONFIG_USED["softplus_scaled_diagnostics_3b"] = softplus_diagnostics
        CONFIG_USED["timings_sec"]["3b"] = round(time.time() - t0_3b, 2)

        print("\n=== 3b NEGATIVE & <=0 FORECAST REPORT SUMMARY ===")
        print(
            df_neg_report.groupby("variant", as_index=False)[
                ["neg_share_before_clip", "le_zero_share_before_clip", "neg_share_after_clip", "le_zero_share_after_clip"]
            ].max().to_string(index=False)
        )
        assert np.all(df_neg_report["le_zero_share_after_clip"].values == 0.0), (
            "All 3b forecasts MUST be strictly positive (> 0)!"
        )
        print(f"Verified 0.00% <=0 predictions across all 3b variants. Saved: {NEG_REPORT_PATH}")
    except Exception as exc:
        CONFIG_USED["timings_sec"]["3b"] = round(time.time() - t0_3b, 2)
        CONFIG_USED["errors"]["3b"] = traceback.format_exc()
        print(f"ERROR in Section 3b (logged to config_used.json):\n{exc}")
else:
    print("Section 3b skipped (RUN_3B=False).")


## Section 4a (Optional `RUN_3C = True`): Expanding 60-Day Rolling Refit for `TFT` (`SPILL` vs `NOSPILL`)

Disabled by default (`RUN_3C = False`). When enabled, retrains `TFT` every `60` test origins (`6` chronological blocks covering `335` origins) for `3` seeds `[42, 7, 13]` (`36` fits total, ~35 min on T4).


In [ ]:
# ------------------------------------------------------------------
# Section 4a (Optional 3c): Expanding 60-Day Rolling Refit for TFT
# ------------------------------------------------------------------
if RUN_3C:
    t0_3c = time.time()
    print("=== SECTION 4a (3c): EXPANDING 60-DAY ROLLING REFIT FOR TFT (36 FITS) ===")
    try:
        blocks = []
        for b_start in range(0, N_TEST_CUTOFFS, BLOCK_SIZE_3C):
            b_end = min(b_start + BLOCK_SIZE_3C, N_TEST_CUTOFFS)
            blocks.append((b_start, b_end))
        print(f"Rolling refit blocks ({len(blocks)}): {blocks}")

        configs_3c = [("TFT", "SPILL", s) for s in SEEDS_3C] + [("TFT", "NOSPILL", s) for s in SEEDS_3C]
        total_3c = len(configs_3c) * len(blocks)

        with tqdm(total=total_3c, desc="Step 3c (TFT Rolling60)", unit="fit", disable=not SHOW_PROGRESS_BAR) as pbar_3c:
            for mname, fset, seed in configs_3c:
                fcols = FEATURE_SETS[fset]
                seed_block_frames = []
                for bi, (b_start, b_end) in enumerate(blocks):
                    pbar_3c.set_postfix_str(f"{fset} s={seed} blk={bi+1}/{len(blocks)}")
                    n_win_blk = b_end - b_start
                    fit_end_pos = base_fit_end + b_start
                    all_end_pos = base_fit_end + b_end + H - 1
                    fit_src_blk = df_post.iloc[:fit_end_pos].reset_index(drop=True)
                    all_src_blk = df_post.iloc[:all_end_pos].reset_index(drop=True)

                    aligned_blk, nf_obj, _, _, fit_entry = fit_and_cv_aligned(
                        model_name=mname,
                        fcols=fcols,
                        fit_src=fit_src_blk,
                        all_src=all_src_blk,
                        target_col="y_r2",
                        seed=seed,
                        feature_set=fset,
                        variant="Rolling60",
                        section_tag=f"3c_blk{bi}",
                        n_windows=n_win_blk,
                        val_size=N_VAL,
                        scaler_type="standard",
                        date_to_true=DATE_TO_R2,
                        expected_origins=range(b_start, b_end),
                    )
                    label = f"3c:{mname}-{fset}:blk{bi}:s{seed}"
                    verify_target_precision_3c(
                        label=label,
                        y_true_f64=aligned_blk["y_true"].values,
                        target_dates=aligned_blk["target_date"].values,
                        nf_y_vals=aligned_blk["nf_y"].values,
                        nf_expected_f64=aligned_blk["y_true"].values,
                        ref_map=DATE_TO_R2,
                    )
                    check_prediction_health_3e(aligned_blk["y_pred"].values, label=label, fit_log_entry=fit_entry)
                    seed_block_frames.append(aligned_blk.drop(columns=["nf_y"]))
                    del nf_obj
                    gc.collect()
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                    pbar_3c.update(1)

                seed_full = pd.concat(seed_block_frames, ignore_index=True).sort_values(["origin_idx", "horizon"]).reset_index(drop=True)
                assert len(seed_full) == N_TEST_CUTOFFS * H
                assert not seed_full.duplicated(["origin_idx", "horizon"]).any()
                ps_3c = PER_SEED_DIR / f"3c_{mname}_{fset}_Rolling60_s{seed}.csv"
                seed_full.to_csv(ps_3c, index=False)
                LONG_FRAMES.append(seed_full)
                tqdm.write(f"  [3c] {mname}-{fset} seed={seed:<2} complete (335 origins) -> {ps_3c.name}")

        CONFIG_USED["timings_sec"]["3c"] = round(time.time() - t0_3c, 2)
        print(f"Section 3c finished in {CONFIG_USED['timings_sec']['3c']/60:.2f} min.")
    except Exception as exc:
        CONFIG_USED["timings_sec"]["3c"] = round(time.time() - t0_3c, 2)
        CONFIG_USED["errors"]["3c"] = traceback.format_exc()
        print(f"ERROR in Section 3c (logged to config_used.json):\n{exc}")
else:
    print("Section 3c skipped (RUN_3C=False).")


## Section 4b (Optional `RUN_3D = True`): Intraday `RV_60m` Forecasting (`TFT` & `NBEATSx`)

Disabled by default (`RUN_3D = False`). When enabled, loads `step3d_indodax_rv60m_daily.csv`, swaps the post-warm-up target to `rv_60m_daily_utc`, verifies zero missing values over the `1,426` post-warm-up dates, and trains `TFT` (`3` seeds) and `NBEATSx` (`5` seeds) across `SPILL` and `NOSPILL` (`16` fits total).


In [ ]:
# ------------------------------------------------------------------
# Section 4b (Optional 3d): Intraday 60-min Realized Variance (RV_60m)
# ------------------------------------------------------------------
HIST_EXP_RV60M_BY_ORIGIN = None
DATE_TO_RV60M = None

if RUN_3D:
    t0_3d = time.time()
    print("=== SECTION 4b (3d): INTRADAY RV_60m DEEP LEARNING (16 FITS) ===")
    try:
        if RV60M_PATH is None:
            raise FileNotFoundError(
                "RUN_3D=True requires step3d_indodax_rv60m_daily.csv attached under /kaggle/input/."
            )
        rv_raw = pd.read_csv(RV60M_PATH)
        rv_raw["Date"] = rv_raw["Date"].astype(str)
        df_rv = df.drop(columns=["rv_60m_daily_utc"], errors="ignore").merge(
            rv_raw[["Date", "rv_60m_daily_utc"]], on="Date", how="left"
        )
        df_rv_post = df_rv.iloc[WARMUP_DISCARD:].reset_index(drop=True)
        assert not df_rv_post["rv_60m_daily_utc"].isna().any(), "Missing rv_60m_daily_utc in post-warmup rows!"
        assert len(df_rv_post) == 1426

        DATE_TO_RV60M = dict(zip(df_rv_post["real_date"].astype(str), df_rv_post["rv_60m_daily_utc"].astype(np.float64)))
        HIST_EXP_RV60M_BY_ORIGIN = np.array(
            [df_rv.loc[df_rv["Date"] <= ALIGNMENT_MANIFEST.loc[o, "cutoff_date"], "rv_60m_daily_utc"].mean()
             for o in range(N_TEST_CUTOFFS)],
            dtype=float,
        )
        fit_rv = df_rv_post.iloc[:n_fit].reset_index(drop=True)

        runs_3d = (
            [("TFT", fset, seed) for fset in ["SPILL", "NOSPILL"] for seed in SEEDS_3D_TFT]
            + [("NBEATSx", fset, seed) for fset in ["SPILL", "NOSPILL"] for seed in SEEDS_3D_NBX]
        )
        with tqdm(runs_3d, desc="Step 3d (Intraday RV60m)", unit="fit", disable=not SHOW_PROGRESS_BAR) as pbar_3d:
            for mname, fset, seed in pbar_3d:
                pbar_3d.set_postfix_str(f"{mname}-{fset} s={seed}")
                fcols = FEATURE_SETS[fset]
                aligned_rv, nf_obj, _, sec, fit_entry = fit_and_cv_aligned(
                    model_name=mname,
                    fcols=fcols,
                    fit_src=fit_rv,
                    all_src=df_rv_post,
                    target_col="rv_60m_daily_utc",
                    seed=seed,
                    feature_set=fset,
                    variant="RV60m",
                    section_tag="3d",
                    n_windows=N_TEST_CUTOFFS,
                    val_size=N_VAL,
                    target_scale=1.0,
                    scaler_type="standard",
                    date_to_true=DATE_TO_RV60M,
                    expected_origins=range(N_TEST_CUTOFFS),
                )
                label = f"3d:{mname}-{fset}:s{seed}"
                verify_target_precision_3c(
                    label=label,
                    y_true_f64=aligned_rv["y_true"].values,
                    target_dates=aligned_rv["target_date"].values,
                    nf_y_vals=aligned_rv["nf_y"].values,
                    nf_expected_f64=aligned_rv["y_true"].values,
                    ref_map=DATE_TO_RV60M,
                )
                check_prediction_health_3e(aligned_rv["y_pred"].values, label=label, fit_log_entry=fit_entry)

                out_rv = aligned_rv.drop(columns=["nf_y"])
                ps_rv = PER_SEED_DIR / f"3d_{mname}_{fset}_RV60m_s{seed}.csv"
                out_rv.to_csv(ps_rv, index=False)
                LONG_FRAMES.append(out_rv)
                tqdm.write(f"  [3d] {mname}-{fset} seed={seed:<2} | time={sec:4.1f}s | saved={ps_rv.name}")

                del nf_obj
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

        CONFIG_USED["timings_sec"]["3d"] = round(time.time() - t0_3d, 2)
        print(f"Section 3d finished in {CONFIG_USED['timings_sec']['3d']/60:.2f} min.")
    except Exception as exc:
        CONFIG_USED["timings_sec"]["3d"] = round(time.time() - t0_3d, 2)
        CONFIG_USED["errors"]["3d"] = traceback.format_exc()
        print(f"ERROR in Section 3d (logged to config_used.json):\n{exc}")
else:
    print("Section 3d skipped (RUN_3D=False).")


## Section 5: Seed-Mean Ensemble Aggregation & Evaluation Metrics

Combines all per-seed outputs into `predictions_long.csv`, averages across seeds by `(model, feature_set, variant, origin_idx, horizon)` into `predictions_ensemble.csv`, checks ensemble prediction health (`3e`), and computes `metrics_by_model.csv` (`MSE`, `MAE`, `QLIKE`, `R2_OOS` vs `Hist-Exp`, `neg_pred_share`, `le_zero_pred_share`, `n_obs`, `mse_seed_sd`) across all `10` horizons.


In [ ]:
# ------------------------------------------------------------------
# Section 5: Aggregate Long -> Ensemble Predictions & Compute Metrics
# ------------------------------------------------------------------
def compute_qlike(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    yt = np.asarray(y_true, dtype=float)
    yp = np.asarray(y_pred, dtype=float)
    ratio = yt / yp
    return ratio - np.log(ratio) - 1.0


t0_eval = time.time()
print("=== SECTION 5: ENSEMBLE AGGREGATION & EVALUATION METRICS ===")
try:
    if not LONG_FRAMES:
        raise RuntimeError("No model predictions were generated (all enabled sections failed or disabled).")

    df_long = pd.concat(LONG_FRAMES, ignore_index=True)
    long_cols = [
        "model", "feature_set", "variant", "seed",
        "origin_idx", "cutoff_date", "target_date", "horizon", "y_true", "y_pred",
    ]
    df_long = (
        df_long[long_cols]
        .sort_values(["model", "feature_set", "variant", "seed", "origin_idx", "horizon"])
        .reset_index(drop=True)
    )
    assert not df_long.duplicated(
        subset=["model", "feature_set", "variant", "seed", "origin_idx", "horizon"]
    ).any(), "Duplicate keys detected in predictions_long!"

    LONG_PATH = OUTPUT_DIR / "predictions_long.csv"
    df_long.to_csv(LONG_PATH, index=False)

    y_true_spread = df_long.groupby(
        ["model", "feature_set", "variant", "origin_idx", "cutoff_date", "target_date", "horizon"]
    )["y_true"].nunique().max()
    assert int(y_true_spread) == 1, "y_true differs across seeds for the same target_date!"

    df_ens = (
        df_long.groupby(
            ["model", "feature_set", "variant", "origin_idx", "cutoff_date", "target_date", "horizon", "y_true"],
            as_index=False,
        )["y_pred"]
        .mean()
        .sort_values(["model", "feature_set", "variant", "origin_idx", "horizon"])
        .reset_index(drop=True)
    )
    ENS_PATH = OUTPUT_DIR / "predictions_ensemble.csv"
    df_ens.to_csv(ENS_PATH, index=False)

    seed_mse_rows = []
    for (m, fs, var, sd, hv), grp in df_long.groupby(["model", "feature_set", "variant", "seed", "horizon"]):
        seed_mse_rows.append({
            "model": m,
            "feature_set": fs,
            "variant": var,
            "seed": int(sd),
            "horizon": int(hv),
            "seed_mse": float(mean_squared_error(grp["y_true"].values, grp["y_pred"].values)),
        })
    df_seed_mse = pd.DataFrame(seed_mse_rows)
    seed_sd_map = (
        df_seed_mse.groupby(["model", "feature_set", "variant", "horizon"])["seed_mse"]
        .std(ddof=1)
        .fillna(0.0)
        .to_dict()
    )

    metric_rows = []
    for (m, fs, var, hv), grp in df_ens.groupby(["model", "feature_set", "variant", "horizon"]):
        grp_s = grp.sort_values("origin_idx").reset_index(drop=True)
        yt = grp_s["y_true"].values.astype(float)
        yp = grp_s["y_pred"].values.astype(float)
        o_idx = grp_s["origin_idx"].values.astype(int)

        hexp_base = HIST_EXP_RV60M_BY_ORIGIN if (var == "RV60m" and HIST_EXP_RV60M_BY_ORIGIN is not None) else HIST_EXP_BY_ORIGIN
        y_hexp = hexp_base[o_idx]

        mse = float(mean_squared_error(yt, yp))
        mae = float(mean_absolute_error(yt, yp))
        mse_hexp = float(mean_squared_error(yt, y_hexp))
        r2_oos = float(1.0 - mse / mse_hexp) if mse_hexp > 0 else np.nan
        qlike = float(np.mean(compute_qlike(yt, yp))) if (np.all(yp > 0.0) and np.all(yt > 0.0)) else np.nan
        neg_share = float(np.mean(yp < 0.0))
        le_zero_share = float(np.mean(yp <= 0.0))
        pred_std = float(np.std(yp))

        metric_rows.append({
            "model": m,
            "feature_set": fs,
            "variant": var,
            "horizon": int(hv),
            "MSE": mse,
            "MAE": mae,
            "QLIKE": qlike,
            "R2_OOS": r2_oos,
            "neg_pred_share": neg_share,
            "le_zero_pred_share": le_zero_share,
            "pred_std": pred_std,
            "n_obs": int(len(yt)),
            "mse_seed_sd": float(seed_sd_map.get((m, fs, var, int(hv)), 0.0)),
        })

    df_metrics = (
        pd.DataFrame(metric_rows)
        .sort_values(["horizon", "variant", "model", "feature_set"])
        .reset_index(drop=True)
    )
    METRICS_PATH = OUTPUT_DIR / "metrics_by_model.csv"
    df_metrics.to_csv(METRICS_PATH, index=False)

    NEG_REPORT_PATH = OUTPUT_DIR / "negative_forecast_report.csv"
    if not NEG_REPORT_PATH.exists():
        pd.DataFrame(columns=[
            "model", "feature_set", "variant", "horizon", "n_obs", "n_seeds",
            "neg_share_before_clip", "le_zero_share_before_clip",
            "neg_share_after_clip", "le_zero_share_after_clip",
        ]).to_csv(NEG_REPORT_PATH, index=False)

    CONFIG_USED["timings_sec"]["metrics_eval"] = round(time.time() - t0_eval, 2)
    print(f"Saved predictions_long.csv ({len(df_long)} rows)")
    print(f"Saved predictions_ensemble.csv ({len(df_ens)} rows)")
    print(f"Saved metrics_by_model.csv ({len(df_metrics)} rows)")

    key_h = df_metrics[df_metrics["horizon"].isin([1, 10])].copy()
    print("\n=== ENSEMBLE METRICS SUMMARY (h = 1 and h = 10) ===")
    print(key_h[[
        "horizon", "model", "feature_set", "variant",
        "MSE", "MAE", "QLIKE", "R2_OOS", "le_zero_pred_share", "pred_std", "mse_seed_sd", "n_obs"
    ]].to_string(index=False))
except Exception as exc:
    CONFIG_USED["timings_sec"]["metrics_eval"] = round(time.time() - t0_eval, 2)
    CONFIG_USED["errors"]["metrics_eval"] = traceback.format_exc()
    print(f"ERROR in Section 5 (logged to config_used.json):\n{exc}")


## Section 6: Configuration Dump (`config_used.json`), Final Verification & Zip Packaging

Saves `config_used.json` (including library versions, SPILL SHA-256, `3c` float64/float32 precision checks, `3e` collapse diagnostics, and timings), packages `/kaggle/working/outputs_3a3b_v2/` into `/kaggle/working/kaggle_outputs_3a3b_v2.zip`, and fails loudly if any error or collapse occurred.


In [ ]:
# ------------------------------------------------------------------
# Section 6: Write config_used.json, Package outputs_3a3b_v2/ -> Zip & Final Status
# ------------------------------------------------------------------
print("=== SECTION 6: CONFIG DUMP & ZIP PACKAGING ===")
CONFIG_PATH = OUTPUT_DIR / "config_used.json"
with open(CONFIG_PATH, "w") as f_cfg:
    json.dump(CONFIG_USED, f_cfg, indent=2, default=str)

expected_files = [
    "predictions_long.csv",
    "predictions_ensemble.csv",
    "metrics_by_model.csv",
    "alignment_manifest.csv",
    "config_used.json",
    "negative_forecast_report.csv",
]

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for fp in sorted(OUTPUT_DIR.rglob("*")):
        if fp.is_file():
            zf.write(fp, arcname=str(fp.relative_to(OUTPUT_DIR.parent)))

def md5_hex(path: Path) -> str:
    h = hashlib.md5()
    with open(path, "rb") as fp:
        while chunk := fp.read(8192):
            h.update(chunk)
    return h.hexdigest()

print(f"\nPackaged archive: {ZIP_PATH} ({ZIP_PATH.stat().st_size / (1024**2):.2f} MB)")
print("\nCore output inventory:")
for fn in expected_files:
    fp = OUTPUT_DIR / fn
    if fp.exists():
        print(f"  [OK] {fn:<32} | {fp.stat().st_size:>10,} bytes | md5={md5_hex(fp)}")
    else:
        print(f"  [MISSING] {fn:<27}")

per_seed_count = len(list(PER_SEED_DIR.glob("*.csv")))
print(f"  [OK] per_seed/*.csv                 | {per_seed_count} files")
print(f"Target precision checks passed: {len(CONFIG_USED['target_precision_checks'])} runs")
print(f"Collapse warnings detected    : {len(CONFIG_USED['collapse_warnings'])}")

if CONFIG_USED["collapse_warnings"]:
    print("\n!!! CRITICAL COLLAPSE WARNINGS RECORDED IN config_used.json !!!")
    for w in CONFIG_USED["collapse_warnings"]:
        print(f"  - {w['run']}: pred_std={w['pred_std']:.3e}, le0={w['le_zero_share']*100:.2f}%")

if CONFIG_USED["errors"]:
    print("\n!!! FAILURE: SECTION ERRORS RECORDED IN config_used.json !!!")
    for sec_name, tb in CONFIG_USED["errors"].items():
        print(f"--- Error in section '{sec_name}' ---\n{tb}")
    raise RuntimeError(f"Notebook run finished with errors in sections: {list(CONFIG_USED['errors'].keys())}")
elif CONFIG_USED["collapse_warnings"]:
    raise RuntimeError("Notebook run detected collapsed predictions (std <= 1e-12).")
else:
    print("\nALL ENABLED SECTIONS COMPLETED CLEANLY WITH ZERO ERRORS AND ZERO COLLAPSES.")
